# Task 7: Physics-Guided ML (PINN)

Same three-class dataset as Task 1, but the model is now physics-informed. An EfficientNet-B3 backbone is shared between two heads:

1. **Classification head** -- produces class logits as usual.
2. **Physics head** -- predicts SIS lens parameters, feeds them into a differentiable lensing layer that reconstructs the observed image from physics first principles.

The reconstruction error becomes a physics residual loss term that regularizes training with knowledge of the gravitational lensing equation.

**Why this helps:** "No substructure" images follow a smooth SIS profile closely, so the physics head can reconstruct them well. Substructure images cannot be reconstructed by a pure SIS model, so the residual carries class-relevant information. The backbone is forced to learn features consistent with lensing physics, not just pattern matching.

**Task 1 reference (EfficientNet-B3, no physics):** Test AUC 0.9778 | Accuracy 89.7%

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt

_here = Path().resolve()
_root = _here.parent if _here.name == "task7" else _here
sys.path.insert(0, str(_root))

from utils.models import PhysicsGuidedEffNet
from utils.losses import LabelSmoothingCrossEntropy

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}  |  AMP: {use_amp}")

## The Gravitational Lensing Equation

The SIS (Singular Isothermal Sphere) is the standard analytic lens model. It relates image-plane position theta to source-plane position beta via:

```
beta = theta - alpha(theta)

SIS deflection:
  alpha(theta) = theta_E * theta / |theta|

where theta_E is the Einstein radius.
```

The physics head predicts four parameters per image: `[theta_E, src_x, src_y, src_sigma]`. The `LensingLayer` applies the SIS equation on a coordinate grid and evaluates a Gaussian source at the deflected positions to produce a reconstructed image. This reconstruction is fully differentiable, so gradients flow back through the lensing equation into the backbone.

```
Total loss = LabelSmoothingCE(logits, labels) + 0.1 * MSE(reconstructed, input)
```

In [ ]:
CLASS_NAMES = ["no", "sphere", "vort"]

class LensingDataset(Dataset):
    def __init__(self, root, split="train", augment=False):
        self.samples, self.augment = [], augment
        split_dir = Path(root) / split
        for label, cls in enumerate(CLASS_NAMES):
            for p in sorted((split_dir / cls).glob("*.npy")):
                self.samples.append((str(p), label))
        counts = np.bincount([l for _, l in self.samples], minlength=3)
        print(f"  {split}: {dict(zip(CLASS_NAMES, counts))}  total={len(self.samples)}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 2: arr = arr[np.newaxis]
        img = torch.from_numpy(arr)
        if self.augment: img = self._augment(img)
        return img, label

    @staticmethod
    def _augment(img):
        if torch.rand(1) < 0.5: img = TF.hflip(img)
        if torch.rand(1) < 0.5: img = TF.vflip(img)
        k = int(torch.randint(0, 4, (1,)))
        if k: img = torch.rot90(img, k, dims=[-2, -1])
        if torch.rand(1) < 0.3:
            c, h, w = img.shape
            sh = int(h * torch.empty(1).uniform_(0.02, 0.15))
            sw = int(w * torch.empty(1).uniform_(0.02, 0.15))
            r0 = int(torch.randint(0, h - sh + 1, (1,)))
            c0 = int(torch.randint(0, w - sw + 1, (1,)))
            img = img.clone()
            img[:, r0:r0+sh, c0:c0+sw] = img.mean()
        return img

def get_sample_weights(ds):
    counts  = np.bincount([l for _, l in ds.samples], minlength=3)
    class_w = 1.0 / (counts.astype(np.float64) + 1e-8)
    return torch.tensor([class_w[l] for _, l in ds.samples])

In [ ]:
torch.manual_seed(42); np.random.seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)

DATA_DIR     = "../task1_data"
EPOCHS       = 50
BATCH_SIZE   = 64
LR_BACKBONE  = 5e-5
LR_HEAD      = 3e-4
WEIGHT_DECAY = 1e-4
SMOOTHING    = 0.05
MIXUP_ALPHA  = 0.2
LAMBDA_PHYS  = 0.1
NUM_WORKERS  = 4
SAVE_DIR     = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

train_ds = LensingDataset(DATA_DIR, "train", augment=True)
val_ds   = LensingDataset(DATA_DIR, "val",   augment=False)
try:    test_ds = LensingDataset(DATA_DIR, "test",  augment=False)
except: test_ds = val_ds; print("No test split, using val")

sample_tensor, _ = train_ds[0]
in_channels = sample_tensor.shape[0]
img_size    = sample_tensor.shape[-1]

sampler = WeightedRandomSampler(get_sample_weights(train_ds), len(train_ds.samples), replacement=True)
pin = device.type == "cuda"
kw  = dict(num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=(NUM_WORKERS > 0))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, **kw)

model = PhysicsGuidedEffNet(in_channels=in_channels, num_classes=3,
                             img_size=img_size, pretrained=True).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"PhysicsGuidedEffNet | params: {n_params:,} | lambda_phys={LAMBDA_PHYS}")

criterion    = LabelSmoothingCrossEntropy(smoothing=SMOOTHING)
param_groups = model.get_param_groups(lr_backbone=LR_BACKBONE, lr_head=LR_HEAD)
optimizer    = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler    = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=[LR_BACKBONE*10, LR_HEAD*10],
    steps_per_epoch=len(train_loader), epochs=EPOCHS,
    pct_start=0.1, anneal_strategy="cos", div_factor=25.0, final_div_factor=1e4,
)
scaler = torch.amp.GradScaler("cuda") if use_amp else None

## Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device,
                    lambda_phys=0.1, mixup_alpha=0.2, scaler=None):
    model.train()
    total_loss, cls_sum, phys_sum, correct, total = 0., 0., 0., 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.amp.autocast("cuda", enabled=(scaler is not None)):
            if mixup_alpha > 0:
                lam = float(np.random.beta(mixup_alpha, mixup_alpha))
                idx = torch.randperm(imgs.size(0), device=device)
                imgs_mix = lam * imgs + (1 - lam) * imgs[idx]
                logits, reconstructed = model(imgs_mix)
                cls_loss = lam * criterion(logits, labels) + (1-lam) * criterion(logits, labels[idx])
            else:
                logits, reconstructed = model(imgs)
                cls_loss = criterion(logits, labels)
            phys_loss = F.mse_loss(reconstructed, imgs)
            loss = cls_loss + lambda_phys * phys_loss
        optimizer.zero_grad()
        if scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        total_loss += loss.item()     * imgs.size(0)
        cls_sum    += cls_loss.item() * imgs.size(0)
        phys_sum   += phys_loss.item()* imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    n = total
    return total_loss/n, cls_sum/n, phys_sum/n, correct/n

@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, correct, total = 0., 0, 0
    all_probs, all_labels = [], []
    tta_t = [lambda x: x, lambda x: TF.hflip(x), lambda x: TF.vflip(x),
             lambda x: torch.rot90(x, 1, dims=[-2,-1]),
             lambda x: torch.rot90(x, 2, dims=[-2,-1]),
             lambda x: torch.rot90(x, 3, dims=[-2,-1])] if tta else [lambda x: x]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if tta:
            probs = torch.stack([
                torch.softmax(model(torch.stack([t(im) for im in imgs]))[0], 1)
                for t in tta_t]).mean(0)
            logits = torch.log(probs + 1e-8)
        else:
            logits, _ = model(imgs); probs = torch.softmax(logits, 1)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        correct    += (probs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        all_probs.append(probs.cpu().numpy()); all_labels.append(labels.cpu().numpy())
    return total_loss/total, correct/total, np.concatenate(all_probs), np.concatenate(all_labels)

def compute_roc_auc(probs, labels):
    y_bin = label_binarize(labels, classes=[0,1,2])
    return float(roc_auc_score(y_bin, probs, multi_class="ovr", average="macro"))

def compute_per_class_auc(probs, labels):
    y_bin = label_binarize(labels, classes=[0,1,2])
    return {CLASS_NAMES[i]: float(roc_auc_score(y_bin[:,i], probs[:,i])) for i in range(3)}

## Training Loop

In [ ]:
best_val_auc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_cls, tr_phys, tr_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        lambda_phys=LAMBDA_PHYS, mixup_alpha=MIXUP_ALPHA, scaler=scaler)
    scheduler.step()
    val_loss, val_acc, val_probs, val_labels = evaluate(model, val_loader, criterion, device)
    val_auc = compute_roc_auc(val_probs, val_labels)
    elapsed = time.time() - t0
    cur_lrs = [pg["lr"] for pg in optimizer.param_groups]
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train loss {tr_loss:.4f} (cls {tr_cls:.4f} phys {tr_phys:.4f}) acc {tr_acc:.4f} | "
          f"Val loss {val_loss:.4f} acc {val_acc:.4f} AUC {val_auc:.4f} | "
          f"LR {cur_lrs[0]:.2e}/{cur_lrs[-1]:.2e} | {elapsed:.1f}s")
    history.append({"epoch": epoch, "train_loss": tr_loss, "train_cls_loss": tr_cls,
                     "train_phys_loss": tr_phys, "train_acc": tr_acc,
                     "val_loss": val_loss, "val_acc": val_acc, "val_auc": float(val_auc)})
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  New best val AUC: {best_val_auc:.4f}")

model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device))
_, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion, device, tta=True)
test_auc  = compute_roc_auc(test_probs, test_labels)
per_class = compute_per_class_auc(test_probs, test_labels)

print(f"\nTest accuracy: {test_acc:.4f}  |  Test ROC-AUC (macro): {test_auc:.4f}")
print("Per-class AUC:", {k: f"{v:.4f}" for k, v in per_class.items()})
print("\nTask 1 reference: accuracy 0.8972 | AUC 0.9778")

results = {"test_accuracy": test_acc, "test_roc_auc": float(test_auc),
           "per_class_auc": per_class, "best_val_auc": best_val_auc, "history": history}
with open(f"{SAVE_DIR}/results.json", "w") as f: json.dump(results, f, indent=2)

## Training Curves

In [ ]:
with open(f"{SAVE_DIR}/results.json") as f: results = json.load(f)
h = results["history"]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot([e["epoch"] for e in h], [e["train_cls_loss"]  for e in h], label="cls")
axes[0].plot([e["epoch"] for e in h], [e["train_phys_loss"] for e in h], label="phys")
axes[0].set_title("Train Loss Components"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([e["epoch"] for e in h], [e["train_loss"] for e in h], label="train")
axes[1].plot([e["epoch"] for e in h], [e["val_loss"]   for e in h], label="val")
axes[1].set_title("Total Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot([e["epoch"] for e in h], [e["train_acc"] for e in h], label="train")
axes[2].plot([e["epoch"] for e in h], [e["val_acc"]   for e in h], label="val")
axes[2].set_title("Accuracy"); axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)

axes[3].plot([e["epoch"] for e in h], [e["val_auc"] for e in h], color="green")
axes[3].axhline(max(e["val_auc"] for e in h), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(e['val_auc'] for e in h):.4f}")
axes[3].axhline(0.9778, color="gray", linestyle=":", alpha=0.7, label="Task1 no-physics 0.9778")
axes[3].set_title("Val ROC-AUC"); axes[3].set_xlabel("Epoch")
axes[3].set_ylim([0.5, 1.0]); axes[3].legend(); axes[3].grid(alpha=0.3)

plt.tight_layout(); plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

y_bin = label_binarize(test_labels, classes=[0,1,2])
fig, ax = plt.subplots(figsize=(6, 5))
for i, (cls, color) in enumerate(zip(CLASS_NAMES, ["steelblue","tomato","seagreen"])):
    fpr, tpr, _ = roc_curve(y_bin[:,i], test_probs[:,i])
    ax.plot(fpr, tpr, color=color, label=f"{cls} (AUC={results['per_class_auc'][cls]:.4f})")
ax.plot([0,1],[0,1],"k--",alpha=0.4)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"PINN ROC Curves (macro AUC={results['test_roc_auc']:.4f})")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/roc_curves.png", dpi=120, bbox_inches="tight"); plt.show()